In [1]:
import pandas as pd
import numpy as np 
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,roc_auc_score, log_loss
from pathlib import Path
import pickle
import optuna
import warnings                           
warnings.filterwarnings('ignore')  

In [2]:
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()

    for path in [current, *current.parents]:
        # Use the data folder to identify the main project directory.
        if (path / 'data').exists():
            return path

    raise FileNotFoundError('Project root not found.')


# Define all important paths used in this notebook.
BASE_DIR = find_project_root(Path.cwd())
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'heart_disease_processed.csv'
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / 'catboost_model.pkl'
COLUMNS_PATH = MODEL_DIR / 'catboost_columns.pkl'

BASE_DIR, DATA_PATH, MODEL_PATH

(WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/data/processed/heart_disease_processed.csv'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/models/catboost_model.pkl'))

In [3]:
data = pd.read_csv(DATA_PATH)
df = data.copy()
df.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40.0,1,0,0.0,0,0,...,1,0,5.0,18.0,15.0,1,0,9.0,4.0,3.0
1,0,0,0,0,25.0,1,0,0.0,1,0,...,0,1,3.0,0.0,0.0,0,0,7.0,6.0,1.0
2,0,1,1,1,28.0,0,0,0.0,0,1,...,1,1,5.0,30.0,30.0,1,0,9.0,4.0,8.0
3,0,1,0,1,27.0,0,0,0.0,1,1,...,1,0,2.0,0.0,0.0,0,0,11.0,3.0,6.0
4,0,1,1,1,24.0,0,0,0.0,1,1,...,1,0,2.0,3.0,0.0,0,0,11.0,5.0,4.0


In [4]:
target_column = 'HeartDiseaseorAttack'

x = df.drop(columns=[target_column])
y = df[target_column]

print(f'X shape: {x.shape}')
print(f'y shape: {y.shape}')

X shape: (229781, 21)
y shape: (229781,)


In [5]:
# Check the class balance before training the model.
class_counts = y.value_counts().sort_index()
class_percentages = y.value_counts(normalize=True).sort_index() * 100

print('Class counts:')
print(class_counts)
print('\nClass percentages:')
print(class_percentages.round(2))

Class counts:
HeartDiseaseorAttack
0    206064
1     23717
Name: count, dtype: int64

Class percentages:
HeartDiseaseorAttack
0    89.68
1    10.32
Name: proportion, dtype: float64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(f'Training set shape: {X_train.shape}, {y_train.shape}')
print(f'Testing set shape: {X_test.shape}, {y_test.shape}')

Training set shape: (183824, 21), (183824,)
Testing set shape: (45957, 21), (45957,)


In [7]:
# Compute the class ratio so XGBoost can give more weight to positive-class errors.
negative_class_count = (y_train == 0).sum()
positive_class_count = (y_train == 1).sum()
scale_pos_weight = negative_class_count / positive_class_count

print(f'scale_pos_weight: {scale_pos_weight:.4f}')

scale_pos_weight: 8.6602


In [8]:
CATboost = CatBoostClassifier(
    iterations=300,          
    learning_rate=0.05,      
    depth=6,                  
    l2_leaf_reg=5,            
    random_seed=42,           
    eval_metric='AUC',        
    loss_function='Logloss', 
    early_stopping_rounds=30,  
    verbose=50,
    scale_pos_weight= scale_pos_weight       
)

In [9]:
CATboost.fit(
    X_train,                           
    y_train,                         
    eval_set=(X_test, y_test),         
    plot=False                         
)

0:	test: 0.8087511	best: 0.8087511 (0)	total: 209ms	remaining: 1m 2s


50:	test: 0.8373575	best: 0.8373575 (50)	total: 1.62s	remaining: 7.91s


100:	test: 0.8392430	best: 0.8392430 (100)	total: 2.88s	remaining: 5.67s


150:	test: 0.8397930	best: 0.8397946 (148)	total: 4.13s	remaining: 4.08s


200:	test: 0.8400565	best: 0.8400565 (200)	total: 5.41s	remaining: 2.66s


250:	test: 0.8402923	best: 0.8402962 (249)	total: 6.66s	remaining: 1.3s


299:	test: 0.8402322	best: 0.8403463 (287)	total: 7.97s	remaining: 0us

bestTest = 0.8403463423
bestIteration = 287

Shrink model to first 288 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=30, eval_metric='AUC', iterations=300, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=42, scale_pos_weight=np.float64(8.660202848284198), verbose=50)

In [10]:
# Generate predictions and prediction probabilities for evaluation.
y_pred = CATboost.predict(X_test)
y_pred_proba = CATboost.predict_proba(X_test)[:, 1]

In [11]:

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC-AUC Score: {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 0.7314
ROC-AUC Score: 0.8403

Confusion Matrix:
[[29835 11434]
 [  908  3780]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.72      0.83     41269
           1       0.25      0.81      0.38      4688

    accuracy                           0.73     45957
   macro avg       0.61      0.76      0.60     45957
weighted avg       0.90      0.73      0.78     45957



In [12]:
study = optuna.create_study(direction="minimize")

def objective(trial):
    # Suggest CatBoost hyperparameters for Optuna to test.
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
        "random_state": 42,
        "eval_metric": "Logloss",
        "verbose": 0
    }

    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]
    loss = log_loss(y_test, y_prob)

    return loss

[I 2026-04-09 03:10:20,254] A new study created in memory with name: no-name-2714c268-9c3c-4609-a1e8-4c19801b1b39


In [13]:
# Run the Optuna search for 50 trials.
study.optimize(objective, n_trials=50)

print("Best Optuna Value:", study.best_value)
print("Best Optuna Params:", study.best_params)


[I 2026-04-09 03:10:25,440] Trial 0 finished with value: 0.30220096844090827 and parameters: {'n_estimators': 221, 'max_depth': 5, 'learning_rate': 0.1363647843627265, 'subsample': 0.7741077259665343, 'scale_pos_weight': 2.8497181860636434}. Best is trial 0 with value: 0.30220096844090827.


[I 2026-04-09 03:10:28,982] Trial 1 finished with value: 0.36874313138110465 and parameters: {'n_estimators': 179, 'max_depth': 4, 'learning_rate': 0.02456159402747406, 'subsample': 0.6615920981288038, 'scale_pos_weight': 4.548814509030439}. Best is trial 0 with value: 0.30220096844090827.


[I 2026-04-09 03:10:32,250] Trial 2 finished with value: 0.48135836419907413 and parameters: {'n_estimators': 219, 'max_depth': 2, 'learning_rate': 0.18972962887689565, 'subsample': 0.522412029997624, 'scale_pos_weight': 8.017141518272474}. Best is trial 0 with value: 0.30220096844090827.


[I 2026-04-09 03:10:36,564] Trial 3 finished with value: 0.2575773670767941 and parameters: {'n_estimators': 267, 'max_depth': 2, 'learning_rate': 0.03328547930833985, 'subsample': 0.6738422331806693, 'scale_pos_weight': 1.223676280589682}. Best is trial 3 with value: 0.2575773670767941.


[I 2026-04-09 03:10:38,568] Trial 4 finished with value: 0.31546807312071484 and parameters: {'n_estimators': 98, 'max_depth': 5, 'learning_rate': 0.026667467029509367, 'subsample': 0.5114787925386954, 'scale_pos_weight': 2.908413691553764}. Best is trial 3 with value: 0.2575773670767941.


...

[I 2026-04-09 03:13:50,340] Trial 46 finished with value: 0.25360226291776305 and parameters: {'n_estimators': 243, 'max_depth': 4, 'learning_rate': 0.14666519913546183, 'subsample': 0.6454224721048706, 'scale_pos_weight': 1.0113935033331194}. Best is trial 46 with value: 0.25360226291776305.


[I 2026-04-09 03:13:56,173] Trial 47 finished with value: 0.2844624172958698 and parameters: {'n_estimators': 224, 'max_depth': 5, 'learning_rate': 0.1688736943305378, 'subsample': 0.6496824509290382, 'scale_pos_weight': 2.330600077130662}. Best is trial 46 with value: 0.25360226291776305.


[I 2026-04-09 03:14:01,123] Trial 48 finished with value: 0.2719465981809408 and parameters: {'n_estimators': 242, 'max_depth': 4, 'learning_rate': 0.29170236397002713, 'subsample': 0.5006162838352392, 'scale_pos_weight': 1.9113298370080234}. Best is trial 46 with value: 0.25360226291776305.


[I 2026-04-09 03:14:05,267] Trial 49 finished with value: 0.2538153380857051 and parameters: {'n_estimators': 191, 'max_depth': 4, 'learning_rate': 0.21516951998699618, 'subsample': 0.66712035071677, 'scale_pos_weight': 1.0020448539444569}. Best is trial 46 with value: 0.25360226291776305.


Best Optuna Value: 0.25360226291776305
Best Optuna Params: {'n_estimators': 243, 'max_depth': 4, 'learning_rate': 0.14666519913546183, 'subsample': 0.6454224721048706, 'scale_pos_weight': 1.0113935033331194}


In [14]:
best = study.best_params

# Train the final CatBoost model using the best Optuna parameters.
tuned_model = CatBoostClassifier(
    **best,
    random_state=42,
    eval_metric="Logloss",
    verbose=0
)
tuned_model.fit(X_train, y_train)

y_pred = tuned_model.predict(X_test)
y_pred_proba = tuned_model.predict_proba(X_test)[:, 1]

print("Best Params:", best)
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC-AUC Score: {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Best Params: {'n_estimators': 243, 'max_depth': 4, 'learning_rate': 0.14666519913546183, 'subsample': 0.6454224721048706, 'scale_pos_weight': 1.0113935033331194}
Accuracy: 0.9010
ROC-AUC Score: 0.8396

Confusion Matrix:
[[40914   355]
 [ 4196   492]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95     41269
           1       0.58      0.10      0.18      4688

    accuracy                           0.90     45957
   macro avg       0.74      0.55      0.56     45957
weighted avg       0.87      0.90      0.87     45957



In [15]:
# Save the trained model and feature columns for later use in prediction.
with open(MODEL_PATH, 'wb') as model_file:
    pickle.dump(CATboost, model_file)

with open(COLUMNS_PATH, 'wb') as columns_file:
    pickle.dump(x.columns.tolist(), columns_file)

print(f'Model saved to: {MODEL_PATH}')
print(f'Columns saved to: {COLUMNS_PATH}')

Model saved to: D:\PROJECT OF DATA SCIENCE\Heart Disease Health Indicators\models\catboost_model.pkl
Columns saved to: D:\PROJECT OF DATA SCIENCE\Heart Disease Health Indicators\models\catboost_columns.pkl


In [16]:
feature_importance_df = pd.DataFrame({
    'feature': x.columns,
    'importance': CATboost.feature_importances_
}).sort_values(by='importance', ascending=False)

feature_importance_df

,feature,importance
18,Age,28.114790
13,GenHlth,17.195378
17,Sex,8.973640
0,HighBP,6.894760
1,HighChol,6.887030
5,Stroke,6.382038
16,DiffWalk,3.561193
4,Smoker,3.067109
20,Income,2.832965
6,Diabetes,2.397489
